# DealScout - Phase 2, Step 1: QLoRa

A introduction to QLoRA (Quantized Low-Rank Adaptation), the technique we use to fine-tune Llama-3.2-3B efficiently on a single GPU.

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datetime import datetime

In [ ]:
# Log in to HuggingFace

load_dotenv(override=True)
login(os.environ["HF_TOKEN"], add_to_git_credential=True)

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "dealscout"
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

LITE_MODE = False

DATA_USER = "Ankit-Singh-ai"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"


## Trying out different Quantization

### Load the Base Model without quantization

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

### Load the Base Model using 8 bit

In [ ]:
quant_config = BitsAndBytesConfig(load_in_8bit=True)

bit_8_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

print(f"Memory footprint: {bit_8_base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
bit_8_base_model

### Load the Tokenizer and the Base Model using 4 bit

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

bit_4_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

print(f"Memory footprint: {bit_4_base_model.get_memory_footprint() / 1e9:,.2f} GB")

In [ ]:
bit_4_base_model

# Understanding LoRA weights and dimensions

## More common QLoRA setup that we'll use for the Lite Training `LITE_MODE=True`

r = 32, which is a common value for r

Target modules are the Attention layers only

In [1]:
# Each of the Target Modules has 2 LoRA Adaptor matrices, called lora_A and lora_B
# These are designed so that weights can be adapted by adding alpha * lora_A * lora_B
# Let's count the number of weights using their dimensions:

r = 32

# See the matrix dimensions above

# Attention layers
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

# Each layer comes to
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

# There are 28 layers
params = lora_layer * 28

# So the total size in MB is
size = (params * 4) / 1_000_000

print(f"Total number of params: {params:,} and size {size:,.1f}MB")

Total number of params: 18,350,080 and size 73.4MB


## And now, for training with the Full Dataset `LITE_MODE=False`

This is a pretty extreme LoRA setup. We have so much training data that I'm trying to train a lot!

We use 256 dimensions in the LoRA matrices and we have LoRA matrices for the attention layers and the MLP layers:

In [2]:
# Each of the Target Modules has 2 LoRA Adaptor matrices, called lora_A and lora_B
# These are designed so that weights can be adapted by adding alpha * lora_A * lora_B
# Let's count the number of weights using their dimensions:

r = 256

# See the matrix dimensions above

# Attention layers
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

# MLP layers
lora_gate_proj = 3072 * r + 8192 * r
lora_up_proj = 3072 * r + 8192 * r
lora_down_proj = 3072 * r + 8192 * r

# Each layer comes to
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj + lora_gate_proj + lora_up_proj + lora_down_proj

# There are 28 layers
params = lora_layer * 28

# So the total size in MB is
size = (params * 4) / 1_000_000

print(f"Total number of params: {params:,} and size {size:,.1f}MB")

Total number of params: 389,021,696 and size 1,556.1MB
